# 2 · Retrain how? New weights, new hyperparameters, or keep training?

When the trigger fires, a retrain can:

- **refit:** same hyperparameters, new weights learned from recent data;
- **retune:** a fresh hyperparameter search on recent data, then fit ("reset weights and hyperparameters");
- **warm start:** keep the old trees and add more, fitted to recent data.

Same 30 streams per scenario as notebook 1; mean prediction error after the change (lower is better).

In [1]:
%matplotlib inline
import os, json, warnings
os.chdir('..')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from retrain import plot
plot.style()
pd.set_option('display.width', 160)
LABELS = {'never': 'never retrain', 'schedule': 'every 10 batches', 'psi': 'input drift: PSI',
          'psi_rule': 'input drift: PSI > 0.2 rule', 'ks': 'input drift: KS test', 'c2st': 'input drift: classifier test',
          'pred_psi': 'prediction drift: PSI', 'error': 'error ratio', 'page_hinkley': 'Page-Hinkley (error)',
          'psi_error': 'input flag + error confirms', 'error_cp': 'error ratio + change point',
          'page_hinkley_cp': 'Page-Hinkley + change point', 'oracle': 'oracle (knows the change)'}

d = pd.read_csv('results/synthetic_runs.csv')

In [2]:
a = d[(d.study == 'actions') | ((d.study == 'triggers') & d.trigger.isin(['error', 'oracle', 'schedule']))]
tab = a.groupby(['scenario', 'trigger', 'action']).error_after_onset.mean().unstack().round(3)
tab[['refit', 'retune', 'warm']]

action                       refit  retune   warm
scenario           trigger                       
concept_complex    error     1.418   1.205  1.590
                   oracle    1.266   0.988  1.444
                   schedule  1.264   0.992  1.367
concept_gradual    error     0.915   0.887  0.922
                   oracle    1.805   1.791  1.802
                   schedule  0.959   0.926  0.965
concept_sudden     error     1.362   1.317  1.377
                   oracle    0.763   0.751  0.771
                   schedule  1.008   1.004  1.017
covariate_harmful  error     0.805   0.781  3.062
                   oracle    0.798   0.750  4.227
                   schedule  0.858   0.843  3.834
covariate_harmless error     0.423   0.423  0.431
                   oracle    0.423   0.423  0.423
                   schedule  0.434   0.431  0.995
none               error     0.422   0.422  0.421
                   oracle    0.421   0.421  0.421
                   schedule  0.436   0.433  0.428

- **Retuning matters when the world gets more complicated.** In *concept_complex*, the settings tuned
  for the simpler world hold the model back; a fresh search cuts the error by about 15-20% for every
  trigger. Elsewhere, retuning helps by only 1-5%: fresh data is what matters.
- **Warm start is the worst option, and dangerous when the inputs move.** After *covariate_harmful*,
  warm-started models are 4-5x worse than refitted ones.

## Why warm start fails here

The warm start only *adds* trees, so in principle it should at least match the old model. A small
experiment: train on inputs centred at 0, then warm-start on inputs shifted by 0, 1 or 4.

In [3]:
from sklearn.ensemble import HistGradientBoostingRegressor as H
rng = np.random.default_rng(0); rows = []
for shift in (0.0, 1.0, 4.0):
    X = rng.normal(0, 1, (3000, 1)); y = X[:, 0]
    m = H(max_iter=50, random_state=0).fit(X, y)
    X2 = rng.normal(shift, 1, (3000, 1)); y2 = X2[:, 0]
    m.set_params(warm_start=True, max_iter=150); m.fit(X2, y2)
    fresh = H(max_iter=150, random_state=0).fit(X2, y2)
    rows.append({'input shift': shift, 'warm start error': np.abs(m.predict(X2) - y2).mean(),
                 'fresh fit error': np.abs(fresh.predict(X2) - y2).mean()})
pd.DataFrame(rows).round(3)

,input shift,warm start error,fresh fit error
0,0.0,0.066,0.008
1,1.0,0.956,0.009
2,4.0,2.583,0.009


The damage grows with the shift. Reading scikit-learn's source explains it: on each `fit`, this model
re-learns how to bucket ("bin") each feature from the *new* data, but while adding trees it evaluates the
*old* trees on the newly bucketed data, whose split points referred to the old buckets. The new trees
learn to correct a wrong starting point; at prediction time the old trees use their original real-valued
splits, so the correction no longer matches. **Practical rule: with this library, warm-start only on data
from the same distribution; after drift, refit.**

## How fast do labels need to come back?

The error-based trigger needs true answers. Same scenarios, labels returned 1 to 10 batches late:

In [4]:
dl = d[(d.study == 'delay') | ((d.study == 'triggers') & (d.trigger == 'error'))]
dl.groupby(['scenario', 'label_delay']).error_total.mean().unstack().round(1)

label_delay,1,2,5,10
scenario,,,,
concept_complex,64.4,65.1,61.8,70.0
concept_gradual,43.3,45.0,49.4,64.5
concept_sudden,61.5,62.9,55.3,71.3
covariate_harmful,39.5,40.6,44.2,56.3
covariate_harmless,25.3,25.3,25.3,25.3
none,25.3,25.3,25.3,25.3


Slower labels cost more, roughly steadily, with one wrinkle: at a 5-batch delay the first retrain happens
later and so trains on *more* post-change data, which can beat a 2-batch delay. The same lesson as the
change-point window: what you retrain on matters as much as when.